# 4.1 - Regression Model: Training and Evaluation

## 1: Dependencies and Imports

In [ ]:
# Instalar LightGBM directamente desde el notebook
%pip install lightgbm

In [ ]:
# Standard library
import os
import time
import warnings

# Third-party
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import BayesianRidge, LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_pinball_loss,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from xgboost.sklearn import XGBRegressor

## 2: Load Data

In [ ]:
# Path
input_file_path = '../data/Filtered.pkl'

if os.path.exists(input_file_path):
    df = pd.read_pickle(input_file_path)
    rows, cols = df.shape
    print(f"Dataset successfully loaded: {rows} rows and {cols} columns.")
else:
    print(f"Error: File not found at {input_file_path}")

# Quick preview of the loaded data
display(df.head(3))

## 3: Data Preparation

In [ ]:
# Feature Selection
drop_columns = ['order_id', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']
X = df.drop(columns=drop_columns + ['is_delayed'], errors='ignore')
y = df['is_delayed']

# Categorical Feature Definintion
categorical_features = [
    'product_category_name_english', 
    'customer_state_num_pred', 
    'seller_state_num_pred',
    'purchase_month', 
    'purchase_day_of_week',
    'is_same_state'
]

for feature in categorical_features:
    if feature in X.columns:
        X[feature] = X[feature].astype('category')

print(f"Encoding completed. Number of predictor variables: {X.shape[1]}")
print(f"Categorical features identified: {categorical_features}")

In [ ]:
# As 'y' (is_delayed) is the binary classification target.
delivery_days = df['actual_delivery_days']

print("Statistical Analysis of Delivery Performance:")
print(f" - Mean Delivery Time:   {delivery_days.mean():.2f} days")
print(f" - Median (50th Percentile): {delivery_days.median():.2f} days")
print(f" - Standard Deviation:   {delivery_days.std():.2f} days")

# Visualizing the distribution to identify skewness and the "long tail" of logistics
plt.figure(figsize=(8, 4))
sns.histplot(delivery_days, bins=40, kde=True, color='purple')

plt.title('Distribution of Actual Delivery Days', fontweight='bold')
plt.xlabel('Days from Purchase to Delivery')
plt.ylabel('Frequency (Order Count)')
plt.xlim(0, 60)  # Limiting the x-axis to focus on the bulk of operations
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Initial split: Reserve 15% for the final Hold-out Test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Second split: Divide the remaining 85% into Training and Validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, stratify=y_train_val, random_state=42
)

print("Data partitioning complete (approx. 72% / 13% / 15%):")
print(f" - Training Set (Train):   {X_train.shape[0]} records")
print(f" - Validation Set (Val):   {X_val.shape[0]} records")
print(f" - Final Test Set (Test):  {X_test.shape[0]} records")

## 4: Model Definition

In [ ]:
warnings.filterwarnings('ignore')

print("Initializing LightGBM model for REGRESSION...")

# Hyperparameter Definition
lgbm_regression_params = {
    'objective': 'regression',          # Changed to regression for continuous value prediction
    'metric': 'rmse',                   # Root Mean Squared Error as the primary loss metric
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'n_estimators': 1500,               
    'num_leaves': 63,                   
    'max_depth': 8,                     
    'min_child_samples': 50,            
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1
}

# Model Instantiation
lgbm_regressor = lgb.LGBMRegressor(**lgbm_regression_params)

In [ ]:
# Model Fitting with Early Stopping
lgbm_regressor.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['Training', 'Validation'],
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, first_metric_only=True, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

print("\nRegression model training complete.")
print(f"Optimal number of iterations (trees): {lgbm_regressor.best_iteration_}")

## 5: Result Calculation

In [ ]:
# Model Evaluation on the Test Set
y_pred = lgbm_regressor.predict(X_test)

# Calculate regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE):    {mae:.2f} days")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} days")
print(f"R-squared (R2 Score):          {r2:.4f}")

# Optional: Display a few sample predictions vs. reality
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).head(10)
print("Sample Predictions:")
print(comparison_df)

In [ ]:
#Pliotting Actual vs. Predicted and Error Distributions
plt.figure(figsize=(14, 6))

# Subplot 1: Actual vs. Predicted Scatter Plot
# This plot visualizes how closely the model's estimations follow reality.
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.2, color='#3498db', s=10)

# Ideal line: where predicted equals actual (y = x)
plt.plot([0, 60], [0, 60], 'r--', lw=2, label="Perfect Prediction")

plt.xlabel("Actual Delivery Days")
plt.ylabel("Model Predicted Days")
plt.title("Actual vs. Predicted Performance", fontweight='bold')
plt.xlim(0, 60)
plt.ylim(0, 60)
plt.legend()
plt.grid(alpha=0.3)

# Subplot 2: Error Distribution Comparison
# Comparing the absolute error of the original system (Olist) vs. our ML Model.
if 'estimated_delivery_margin_days' in X_test.columns:
    plt.subplot(1, 2, 2)
    
    # Calculate Absolute Error for both systems
    olist_absolute_error = np.abs(y_test - X_test['estimated_delivery_margin_days'])
    model_absolute_error = np.abs(y_test - y_pred)
    
    # Use KDE plots to visualize the density of error magnitudes
    sns.kdeplot(olist_absolute_error, fill=True, color="red", label="Olist (Baseline) Error", alpha=0.4)
    sns.kdeplot(model_absolute_error, fill=True, color="blue", label="ML Model (LightGBM) Error", alpha=0.4)
    
    plt.title('Error Magnitude Comparison', fontweight='bold')
    plt.xlabel('Absolute Error (Days)')
    plt.ylabel('Density')
    plt.xlim(0, 30)
    plt.legend()
    plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6: Advance Metrics and Performance

In [ ]:
# Evaluating Olist's Legacy Estimation Performance
print("BASELINE ANALYSIS: OLIST ESTIMATION ERROR\n")
print("-" * 50)

# 1. Calculate the raw estimation error in days
# Error = Actual Days - Estimated Days
# Negative result: The package arrived BEFORE the deadline (Early).
# Positive result: The package arrived AFTER the deadline (Delayed).
df['olist_error_days'] = df['actual_delivery_days'] - df['estimated_delivery_margin_days']

# 2. Compute Business Metrics
# Mean Absolute Error (MAE): The average magnitude of error, regardless of direction.
mae_baseline = df['olist_error_days'].abs().mean()

# Mean Bias: Indicates if the system is systematically pessimistic or optimistic.
# If negative, the system overestimates delivery times to "play it safe."
mean_bias = df['olist_error_days'].mean()

# Operational Breakdown
early_percentage = (df['olist_error_days'] < 0).mean() * 100
on_time_percentage = (df['olist_error_days'] == 0).mean() * 100
late_percentage = (df['olist_error_days'] > 0).mean() * 100

print(f"Olist Baseline Mean Absolute Error (MAE): {mae_baseline:.2f} days")
print(f"Estimation Bias: {mean_bias:.2f} days (Average deviation from reality)")

print("\nCustomer Promise Breakdown:")
print(f" - Delivered Early:   {early_percentage:.2f}% (Customer surprise)")
print(f" - Delivered On-Time: {on_time_percentage:.2f}% (Exact precision)")
print(f" - Delivered Late:    {late_percentage:.2f}% (Customer friction/delays)")
print("-" * 50)

# 3. Visualization of the Residual Distribution
plt.figure(figsize=(12, 6))

# Histogram with KDE to visualize the concentration and spread of the error
sns.histplot(df['olist_error_days'], bins=80, kde=True, color='teal')

# Reference Lines
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (Error 0)')
plt.axvline(x=mean_bias, color='orange', linestyle=':', linewidth=2, label=f'Mean Bias ({mean_bias:.1f} days)')

plt.title('Olist Estimation Error Distribution (Residual Analysis)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Error in Days (Negative = Early | Positive = Late)', fontsize=12)
plt.ylabel('Order Count', fontsize=12)

# Zoom into the most relevant data range to avoid extreme outliers distorting the view
plt.xlim(-30, 20) 
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:

print("🏆 COMPARATIVA DEFINITIVA: OLIST vs INTELIGENCIA ARTIFICIAL\n")
print("-" * 50)

# 1. Extraer las estimaciones originales de Olist para el conjunto de Test
y_olist_pred = X_test['estimated_delivery_margin_days']

# 2. Calcular MAE y RMSE manualmente para Olist
mae_olist = mean_absolute_error(y_test, y_olist_pred)
rmse_olist = np.sqrt(mean_squared_error(y_test, y_olist_pred))

# 3. Recalcular MAE y RMSE para el Modelo IA (por si no estaban en memoria)
mae_ml = mean_absolute_error(y_test, y_pred)
rmse_ml = np.sqrt(mean_squared_error(y_test, y_pred))

# 4. Mostrar resultados numéricos
print(f"🏢 MÉTRICAS DE LA EMPRESA (Estimación Olist):")
print(f" - MAE:  {mae_olist:.2f} días")
print(f" - RMSE: {rmse_olist:.2f} días\n")

print(f"🤖 MÉTRICAS DEL MODELO (LightGBM):")
print(f" - MAE:  {mae_ml:.2f} días")
print(f" - RMSE: {rmse_ml:.2f} días\n")

# Calcular porcentaje de mejora
mejora_mae = ((mae_olist - mae_ml) / mae_olist) * 100
print(f"🔥 Impacto de Negocio: El modelo reduce el error de estimación en un {mejora_mae:.1f}%.")
print("-" * 50)

# 5. Visualización Gráfica para Presentación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- GRÁFICO 1: Barras Comparativas (El resumen ejecutivo) ---
metricas = ['MAE (Error Promedio)', 'RMSE (Penaliza Errores Graves)']
valores_olist = [mae_olist, rmse_olist]
valores_ml = [mae_ml, rmse_ml]

x = np.arange(len(metricas))
width = 0.35

# Barras (Rojo para el baseline actual, Verde para tu mejora)
axes[0].bar(x - width/2, valores_olist, width, label='Olist (Actual)', color='#e74c3c', alpha=0.9)
axes[0].bar(x + width/2, valores_ml, width, label='Modelo IA', color='#2ecc71', alpha=0.9)

axes[0].set_ylabel('Margen de Error (Días)', fontweight='bold')
axes[0].set_title('Comparativa de Error Global', fontweight='bold', fontsize=14)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metricas, fontweight='bold', fontsize=12)
axes[0].legend(fontsize=12)

# Añadir los números exactos sobre cada barra
for i, v in enumerate(valores_olist):
    axes[0].text(i - width/2, v + 0.3, f"{v:.2f}", ha='center', va='bottom', fontweight='bold', fontsize=11)
for i, v in enumerate(valores_ml):
    axes[0].text(i + width/2, v + 0.3, f"{v:.2f}", ha='center', va='bottom', fontweight='bold', fontsize=11)


# --- GRÁFICO 2: Densidad del Error Absoluto (El detalle técnico) ---
# Calculamos la desviación absoluta en días por cada pedido individual
error_abs_olist = np.abs(y_test - y_olist_pred)
error_abs_ml = np.abs(y_test - y_pred)

sns.kdeplot(error_abs_olist, fill=True, color="#e74c3c", label="Error Olist", alpha=0.4, ax=axes[1], linewidth=2)
sns.kdeplot(error_abs_ml, fill=True, color="#2ecc71", label="Error Modelo IA", alpha=0.4, ax=axes[1], linewidth=2)

axes[1].set_title('Distribución del Error por Pedido', fontweight='bold', fontsize=14)
axes[1].set_xlabel('¿Por cuántos días se equivocó? (Error Absoluto)', fontweight='bold')
axes[1].set_ylabel('Concentración de Pedidos')
axes[1].set_xlim(0, 25) # Acotamos a 25 días para ver el grueso de la campana
axes[1].legend(fontsize=12)
axes[1].grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 7: Save Model

In [ ]:
# Ensure the directory exists
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Define the model path; joblib is often more efficient for large NumPy arrays
model_file_path = os.path.join(models_dir, 'lgbm_regression_model.joblib')

# Serialize the model
joblib.dump(lgbm_regressor, model_file_path)

print(f"Model successfully saved to: {model_file_path}")